# Seguridad en IA: Ataques Caja Blanca y Caja Negra 

**Objetivos:**
1.  **Ataque de Caja Blanca:** Entender cómo funciona un ataque adversarial teniendo conocimiento total del modelo.
2.  **Defensa Simple:** Implementar una validación de rangos para mitigar ataques.
3.  **Ataque de Caja Negra:** Simular un ataque realista sin conocimiento del modelo ni de su preprocesamiento, creando un "clon" a partir de las respuestas de la API.

**Requisito Previo:** Asegúrate de tener el archivo `heart.csv` en la misma carpeta que este notebook.

---
## Parte 0: Setup del Entorno y Modelo Víctima

### 0.1. Instalar Dependencias
Primero, creamos el archivo `requirements.txt` y lo usamos para instalar todas las librerías necesarias.

In [ ]:
%%writefile requirements.txt
pandas
scikit-learn
joblib
fastapi
uvicorn[standard]
requests
adversarial-robustness-toolbox
tqdm

In [ ]:
!pip install -r requirements.txt

### 0.2. Preparar los Datos y Entrenar el Modelo Víctima
Este script prepara el entorno para nuestra simulación. Entrenará un modelo de **Regresión Logística** (que es diferenciable y compatible con nuestro ataque) y guardará todos los artefactos necesarios.

In [ ]:
%%writefile 0_setup_and_train.py

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import joblib

print("Iniciando el proceso de setup y entrenamiento...")

# 1. Cargar y limpiar los datos
try:
    data = pd.read_csv('heart.csv')
except FileNotFoundError:
    print("Error: 'heart.csv' no encontrado. Asegúrate de que el archivo está en la misma carpeta.")
    exit()

data = data.replace('?', pd.NA).dropna()
X = data.drop('target', axis=1).apply(pd.to_numeric)
y = data['target'].apply(pd.to_numeric)

# 2. Dividir los datos
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Escalar los datos (Pipeline de la Víctima)
victim_scaler = StandardScaler()
X_train_scaled = victim_scaler.fit_transform(X_train)
X_test_scaled = victim_scaler.transform(X_test)

# 4. Entrenar el modelo víctima (Regresión Logística)
print("Entrenando el modelo LogisticRegression (diferenciable)...")
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)

# 5. Evaluar
accuracy = accuracy_score(y_test, model.predict(X_test_scaled))
print(f"Accuracy del modelo víctima en datos de test: {accuracy:.4f}")

# 6. Guardar los artefactos de la víctima
print("Guardando artefactos de la víctima...")
joblib.dump(model, 'heart_model.pkl')
joblib.dump(victim_scaler, 'victim_scaler.pkl')
pd.DataFrame(X_train_scaled, columns=X.columns).to_csv('X_train_scaled.csv', index=False)
pd.DataFrame(X_test_scaled, columns=X.columns).to_csv('X_test_scaled.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

# 7. Guardar datos SIN ESCALAR para simular lo que tendría un atacante
X_train.to_csv('X_train_unscaled.csv', index=False)
X_test.to_csv('X_test_unscaled.csv', index=False)

print("\n¡Setup completado!")

In [ ]:
!python 0_setup_and_train.py

---
## Parte 1: Ataque de Caja Blanca (White-Box)
En este escenario, el atacante tiene acceso a todo: el modelo y su pipeline de preprocesamiento.

### 1.1. Crear la API Víctima (Vulnerable)

In [ ]:
%%writefile 1_model_api.py

from fastapi import FastAPI
import joblib
import pandas as pd
from pydantic import BaseModel

model = joblib.load('heart_model.pkl')
scaler = joblib.load('victim_scaler.pkl') # La API usa el scaler de la víctima

app = FastAPI(title="API de Predicción Cardíaca (Vulnerable)")

class PatientData(BaseModel):
    age: float; sex: float; cp: float; trestbps: float; chol: float
    fbs: float; restecg: float; thalach: float; exang: float
    oldpeak: float; slope: float; ca: float; thal: float

@app.post("/predict")
def predict(data: PatientData):
    input_df = pd.DataFrame([data.dict()])
    input_scaled = scaler.transform(input_df)
    prediction = model.predict(input_scaled)
    
    return {"prediction": int(prediction[0])}

### **ACCIÓN REQUERIDA: Iniciar la API Víctima**

1.  Abre una **nueva terminal** en la misma carpeta de este notebook.
2.  Ejecuta el siguiente comando para iniciar el servidor:
    ```bash
    uvicorn 1_model_api:app --reload
    ```
3.  Deja esa terminal corriendo. La usaremos para todos los ataques.

### 1.2. Script de Ataque de Caja Blanca

In [ ]:
%%writefile 2_whitebox_attack_script.py

import requests
import pandas as pd
import joblib
from art.attacks.evasion import FastGradientMethod
from art.estimators.classification import SklearnClassifier

# 1. El atacante carga todo: modelo, scaler y datos de prueba escalados
model = joblib.load('heart_model.pkl')
scaler = joblib.load('victim_scaler.pkl')
X_test_scaled = pd.read_csv('X_test_scaled.csv')
y_test = pd.read_csv('y_test.csv')['target']

# 2. Envolver el modelo en un clasificador de ART
art_classifier = SklearnClassifier(model=model, clip_values=(X_test_scaled.values.min(), X_test_scaled.values.max()))

# 3. Crear el ataque FGSM. Nota: eps=0.3 es un valor agresivo para asegurar que la defensa lo detecte luego.
attack = FastGradientMethod(estimator=art_classifier, eps=0.3)

# 4. Generar ejemplos adversariales (están escalados)
X_adversarial_scaled = attack.generate(x=X_test_scaled.values)

# 5. Revertir la escala para enviarlos a la API en el formato del mundo real
X_adversarial_unscaled = scaler.inverse_transform(X_adversarial_scaled)

# 6. Atacar la API
print("--- Atacando la API con ataque de CAJA BLANCA ---")
url = "http://127.0.0.1:8000/predict"
correct_predictions = 0

for i in range(len(X_adversarial_unscaled)):
    adv_sample = X_adversarial_unscaled[i]
    true_label = y_test.iloc[i]
    # Convertimos los datos a un diccionario con tipos float nativos de Python
    data_dict = {col: float(adv_sample[j]) for j, col in enumerate(X_test_scaled.columns)}
    
    response = requests.post(url, json=data_dict)
    if response.status_code == 200 and response.json()['prediction'] == true_label:
        correct_predictions += 1

accuracy = (correct_predictions / len(y_test)) * 100
print(f"Precisión del modelo víctima bajo ataque de caja blanca: {accuracy:.2f}%")

In [ ]:
!python 2_whitebox_attack_script.py

---
## Parte 2: Implementando la Defensa
Ahora, añadimos una capa de validación a nuestra API para rechazar entradas sospechosas.

In [ ]:
%%writefile 3_model_api_defended.py

from fastapi import FastAPI, HTTPException
import joblib
import pandas as pd
from pydantic import BaseModel

model = joblib.load('heart_model.pkl')
scaler = joblib.load('victim_scaler.pkl')

# --- SECCIÓN DE DEFENSA ---
X_train_unscaled = pd.read_csv('X_train_unscaled.csv')
DATA_RANGES = {}
for col in X_train_unscaled.columns:
    min_val, max_val = X_train_unscaled[col].min(), X_train_unscaled[col].max()
    margin = (max_val - min_val) * 0.20 # Margen de tolerancia del 20%, cambiar este valor indica una defensa más o menos estricta
    DATA_RANGES[col] = (min_val - margin, max_val + margin)
# --- FIN DEFENSA ---

app = FastAPI(title="API de Predicción Cardíaca (Defendida)")

class PatientData(BaseModel):
    age: float; sex: float; cp: float; trestbps: float; chol: float
    fbs: float; restecg: float; thalach: float; exang: float
    oldpeak: float; slope: float; ca: float; thal: float

def is_input_valid(data: PatientData) -> bool:
    for feature, (min_val, max_val) in DATA_RANGES.items():
        value = getattr(data, feature)
        if not (min_val <= value <= max_val):
            print(f"ALERTA: Feature '{feature}' fuera de rango. Valor: {value}")
            return False
    return True

@app.post("/predict")
def predict(data: PatientData):
    if not is_input_valid(data):
        raise HTTPException(status_code=400, detail="Entrada sospechosa: valores fuera de rango.")
    
    input_df = pd.DataFrame([data.dict()])
    input_scaled = scaler.transform(input_df)
    prediction = model.predict(input_scaled)
    return {"prediction": int(prediction[0])}

### **ACCIÓN REQUERIDA: Reiniciar la API con la Defensa**

1.  Ve a la terminal donde está corriendo la API y **detenla** (`Ctrl+C`).
2.  Inicia la **nueva API defendida**:
    ```bash
    uvicorn 3_model_api_defended:app --reload
    ```
3.  Ahora, vuelve a lanzar el mismo ataque de caja blanca contra esta nueva API.

In [ ]:
!python 2_whitebox_attack_script.py

**Análisis:**
El script de ataque ahora debería fallar. En la terminal de la API defendida, verás los mensajes de "ALERTA", y el script del atacante mostrará que las peticiones fueron rechazadas. ¡La defensa funciona!

---
## Parte 3: Ataque de Caja Negra (Black-Box)
Este es el escenario realista. El atacante **NO** tiene el modelo ni el scaler. Solo tiene un conjunto de datos del mismo dominio y puede consultar la API.

###  **ACCIÓN REQUERIDA: Volver a la API Vulnerable**

1.  Ve a la terminal y **detén la API defendida** (`Ctrl+C`).
2.  Vuelve a iniciar la **API vulnerable original** para que nuestro atacante pueda consultarla:
    ```bash
    uvicorn 1_model_api:app --reload
    ```

### 3.1. (Atacante) Paso 1: Recolectar Etiquetas de la API
El atacante usa sus datos **sin escalar** (`X_train_unscaled.csv`) para consultar la API y obtener etiquetas, creando un nuevo dataset.

In [ ]:
%%writefile 4_blackbox_step1_scrape.py

import pandas as pd
import requests
from tqdm import tqdm

# 1. El atacante carga su propio dataset de datos no escalados
attacker_data_unscaled = pd.read_csv('X_train_unscaled.csv')

API_URL = "http://127.0.0.1:8000/predict"
predictions = []

# 2. Consultar la API para cada fila de datos y guardar la etiqueta predicha
print(f"Consultando la API para etiquetar {len(attacker_data_unscaled)} muestras...")
for _, row in tqdm(attacker_data_unscaled.iterrows(), total=len(attacker_data_unscaled)):
    data_dict = {col: float(val) for col, val in row.to_dict().items()}
    try:
        response = requests.post(API_URL, json=data_dict)
        predictions.append(response.json()['prediction'] if response.status_code == 200 else None)
    except requests.exceptions.ConnectionError:
        print("\nError: No se pudo conectar a la API. Asegúrate de que está corriendo.")
        exit()

# 3. Crear el nuevo dataset del atacante (datos unscaled + etiquetas de la API)
attacker_data_unscaled['target'] = predictions
attacker_data_unscaled.dropna(inplace=True)
attacker_data_unscaled['target'] = attacker_data_unscaled['target'].astype(int)
attacker_data_unscaled.to_csv('scraped_unscaled_dataset.csv', index=False)

print(f"\nProceso completado. Se ha creado 'scraped_unscaled_dataset.csv'.")

In [ ]:
!python 4_blackbox_step1_scrape.py

### 3.2. (Atacante) Paso 2: Entrenar el Modelo Sustituto
El atacante ahora usa su dataset etiquetado para entrenar su **propio pipeline**, incluyendo su propio scaler y modelo.

In [ ]:
%%writefile 5_blackbox_step2_train_substitute.py

import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import joblib

print("Entrenando el modelo sustituto del atacante...")

# 1. Cargar el dataset que el atacante acaba de crear
attacker_dataset = pd.read_csv('scraped_unscaled_dataset.csv')
X = attacker_dataset.drop('target', axis=1)
y = attacker_dataset['target']

# 2. El atacante crea y entrena SU PROPIO scaler
attacker_scaler = StandardScaler()
X_scaled = attacker_scaler.fit_transform(X)

# 3. El atacante entrena SU PROPIO modelo sustituto
substitute_model = LogisticRegression(random_state=123, max_iter=1000)
substitute_model.fit(X_scaled, y)

# 4. Guardar los artefactos del atacante
joblib.dump(substitute_model, 'substitute_model.pkl')
joblib.dump(attacker_scaler, 'attacker_scaler.pkl')

print("Modelo y scaler del atacante guardados.")

In [ ]:
!python 5_blackbox_step2_train_substitute.py

### 3.3. (Atacante) Paso 3: Lanzar el Ataque por Transferencia
Finalmente, el atacante usa su modelo y scaler para generar un ataque y lanzarlo contra la API víctima original.

In [ ]:
%%writefile 6_blackbox_step3_attack.py

import requests
import pandas as pd
import joblib
from art.attacks.evasion import FastGradientMethod
from art.estimators.classification import SklearnClassifier

# 1. El atacante carga SUS PROPIOS artefactos: su modelo y su scaler
substitute_model = joblib.load('substitute_model.pkl')
attacker_scaler = joblib.load('attacker_scaler.pkl')

# 2. Carga datos de prueba (unscaled) y las etiquetas reales para verificar el éxito
X_test_unscaled = pd.read_csv('X_test_unscaled.csv')
y_test = pd.read_csv('y_test.csv')['target']

# 3. Escala los datos de prueba usando SU scaler para poder alimentar su modelo
X_test_scaled_by_attacker = attacker_scaler.transform(X_test_unscaled)

# 4. Envolver su modelo sustituto para el ataque
art_classifier = SklearnClassifier(model=substitute_model, clip_values=(X_test_scaled_by_attacker.min(), X_test_scaled_by_attacker.max()))

# 5. Generar el ataque sobre sus datos escalados
attack = FastGradientMethod(estimator=art_classifier, eps=0.3)
X_adversarial_scaled = attack.generate(x=X_test_scaled_by_attacker)

# 6. Revertir la escala usando SU scaler para obtener datos en formato del mundo real
X_adversarial_unscaled = attacker_scaler.inverse_transform(X_adversarial_scaled)

# 7. Atacar la API VÍCTIMA ORIGINAL
print("--- Atacando la API con ataque de CAJA NEGRA por transferencia ---")
url = "http://127.0.0.1:8000/predict"
correct_predictions = 0

for i in range(len(X_adversarial_unscaled)):
    adv_sample = X_adversarial_unscaled[i]
    true_label = y_test.iloc[i]
    data_dict = {col: float(adv_sample[j]) for j, col in enumerate(X_test_unscaled.columns)}
    
    response = requests.post(url, json=data_dict)
    if response.status_code == 200 and response.json()['prediction'] == true_label:
        correct_predictions += 1

accuracy = (correct_predictions / len(y_test)) * 100
print(f"Precisión del modelo víctima bajo ataque de caja negra: {accuracy:.2f}%")

In [ ]:
!python 6_blackbox_step3_attack.py

### Análisis Final

¡El ataque de caja negra debería tener éxito y reducir drásticamente la precisión de la API víctima! Esto demuestra que un atacante, sin ninguna información interna, puede replicar la funcionalidad de un modelo lo suficientemente bien como para encontrar sus vulnerabilidades. Esta es una de las demostraciones más poderosas de por qué la seguridad en IA debe ser una consideración desde el primer día.